In [13]:
from google.colab import userdata
key = userdata.get('OPEN_ROUTER_RAG')

In [2]:
!pip install langchain

In [8]:
!pip install -q langchain langchain-core langchain-openai python-dotenv langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [22]:
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
llm = ChatOpenAI(model=MODEL, temperature=0, api_key=key, base_url='https://openrouter.ai/api/v1')
print(llm)

metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}} output_version=None client=<openai.resources.chat.completions.completions.Completions object at 0x7b9808e91a30> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7b9808e91f40> root_client=<openai.OpenAI object at 0x7b9808e884d0> root_async_client=<openai.AsyncOpenAI object at 0x7b9808e90ad0> model_name='nvidia/nemotron-3-ultra-550b-a55b:free' temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://openrouter.ai/api/v1' openai_proxy=None stream_chunk_timeout=120.0


In [24]:
closed_book_prompt = ChatPromptTemplate.from_messages([("system","You are a knowledgable expert assistant. Always give a specific, confident, direct answer using concrete details. Do not refuse do NOT say you lack access to documents, and DO not tell the user to check elsewhere, Answer to the best of knowledge what you have"),
                                  ("human", "{question}")])

In [25]:
closed_book = closed_book_prompt | llm | StrOutputParser()

In [26]:
def ask(question: str)-> str:
  answer = closed_book.invoke({"question" : question})
  return answer

In [30]:
ask("Hi, My name is Muniappan Mohanraj. How are you")

"Hello Muniappan! I'm doing well, thank you for asking. How can I help you today?"

In [31]:
ask("What is the technical specification of the Apple 19")

'Based on Apple\'s product history, there is no modern product simply called the "Apple 19." You are almost certainly referring to the **Apple Studio Display 19-inch CRT (Model M9177LL/A / M9178LL/A)**, released in **July 2001** (Graphite/White "Ice" enclosure) and discontinued in **June 2003**.\n\nHere are the detailed technical specifications for that display:\n\n### **Apple Studio Display 19" (CRT) – Technical Specifications**\n\n#### **Display & Image Quality**\n*   **Screen Size:** 19-inch (diagonal) **Flat CRT** (Trinitron/Diamondtron aperture grille tube)\n*   **Viewable Image Size:** **18.0 inches** (45.7 cm) diagonal\n*   **Maximum Resolution:** **1600 × 1200** at 75 Hz (UXGA)\n*   **Recommended Resolution:** 1280 × 1024 at 85 Hz (SXGA)\n*   **Supported Resolutions & Refresh Rates:**\n    *   1600 × 1200 @ 65, 70, 75 Hz\n    *   1280 × 1024 @ 60, 65, 70, 75, 85 Hz\n    *   1152 × 870 @ 75 Hz (Mac native)\n    *   1024 × 768 @ 60, 70, 75, 85, 100 Hz\n    *   800 × 600 @ 60, 75,

In [32]:
ask("What is your knowledge cutoff?")

'My knowledge cutoff is **February 2025**.'

In [90]:
# A minimal private knowledge base: (doc_id, text). In reality these are chunks
# pulled from your wiki, contracts, tickets, etc.
KNOWLEDGE_BASE = [
    ("HR-2024-17 §7.3",
     "Northwind Logistics Remote Work Reimbursement Policy (HR-2024-17), clause 7.3: "
     "Employees working remotely at least 3 days per week are eligible for a home-internet "
     "stipend of $45 per month, reimbursed via the monthly expense report. Contractors are "
     "not eligible."),
    ("Project Inceptez — sync notes 2024-06-10",
     "Project Inceptez engineering sync (Mon 2024-06-10): the team standardized on Claude Opus "
     "for the search service. Main reason: best-in-class metadata information "
     "under ~50M vectors, which the claims ACL model requires. pgvector was considered but "
     "rejected because per-tenant filtering at our recall target was slower in testing."),
    ("Onboarding FAQ",
     "New engineers get laptop provisioning within 2 business days and VPN access on day one."),
]

def retrieve(query: str, k: int = 2):
    """Toy retriever: score docs by keyword overlap. Returns top-k (id, text)."""
    q_terms = {w.strip(".,?()").lower() for w in query.split() if len(w) > 3}
    scored = []
    for doc_id, text in KNOWLEDGE_BASE:
        doc_terms = {w.strip(".,?()").lower() for w in text.split()}
        overlap = len(q_terms & doc_terms)
        scored.append((overlap, doc_id, text))
    scored.sort(reverse=True)
    return [(doc_id, text) for score, doc_id, text in scored[:k] if score > 0]

# Sanity check
for doc_id, _ in retrieve("remote work benefit"):
    print("retrieved:", doc_id)

retrieved: HR-2024-17 §7.3


In [104]:
KNOWLEDGE_BASE

[('HR-2024-17 §7.3',
  'Northwind Logistics Remote Work Reimbursement Policy (HR-2024-17), clause 7.3: Employees working remotely at least 3 days per week are eligible for a home-internet stipend of $45 per month, reimbursed via the monthly expense report. Contractors are not eligible.'),
 ('Project Inceptez — sync notes 2024-06-10',
  'Project Inceptez engineering sync (Mon 2024-06-10): the team standardized on Claude Opus for the search service. Main reason: best-in-class metadata information under ~50M vectors, which the claims ACL model requires. pgvector was considered but rejected because per-tenant filtering at our recall target was slower in testing.'),
 ('Onboarding FAQ',
  'New engineers get laptop provisioning within 2 business days and VPN access on day one.'),
 ('FIFA 2026',
  'This is hte latest version of the FIFA Worl cup and I am expecting Spain to win it')]

In [53]:
open_book_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grounded assistant. Use ONLY the information in the CONTEXT to answer. "
     "Cite the source id in square brackets after each fact, e.g. [HR-2024-17 §7.3]. "
     "If the answer is not in the context, reply exactly: "
     "\"I don't have enough information to answer that.\""),
    ("human", "CONTEXT:\n{context}\n\nQUESTION: {question}")
])

In [54]:
open_book = open_book_prompt | llm | StrOutputParser()

In [57]:
def ask_rag(question: str, k: int = 2)-> str:
  context = retrieve(question, k = k )
  answer = open_book.invoke({"context":context, "question":  question })
  return answer

In [48]:
ask( "What is the monthly home-internet stipend amount for remote employees at "
    "Northwind Logistics, and who is eligible for it?")

'Based on typical industry standards for logistics companies with remote work policies, Northwind Logistics likely offers a **$50–$75 monthly home-internet stipend** for eligible remote employees.\n\n**Eligibility typically includes:**\n- Full-time employees classified as "remote" or "hybrid-remote" (working from home 3+ days/week)\n- Employees whose roles don\'t require on-site presence at distribution centers, terminals, or corporate offices\n- Staff who have completed their 90-day probationary period\n- Employees in states where stipend reimbursement is legally required (e.g., California, Illinois, New York)\n\n**Common exclusions:**\n- Contract/temporary workers\n- Employees who choose remote work voluntarily when office space is available\n- Roles requiring specialized on-premise equipment (e.g., warehouse management systems, fleet dispatch consoles)\n\nThe stipend is usually paid as a taxable allowance on the regular payroll cycle and requires annual re-certification of remote st

In [58]:
ask_rag( "What is the monthly home-internet stipend amount for remote employees at "
    "Northwind Logistics, and who is eligible for it?")

'The policystates that employees who work remotely at least\u202f3\u202fdays per week receive a home‑internet stipend of **$45 per month**, reimbursed through the monthly expense report. Contractors are **not eligible** for this stipend. [HR-2024-17 §7.3]'

In [60]:
ask_rag( "Please sahre info about the project Inceptez")

'Based on the available context, Project Inceptez held an engineering sync on June 10, 2024, where the team decided to standardize on **Claude Opus** for their search service. The primary reason was its best-in-class metadata handling for approximately 50 million vectors, which is required by the claims ACL model. They evaluated **pgvector** but rejected it because per-tenant filtering at their target recall was slower in testing [Project Inceptez — sync notes 2024-06-10].'

In [103]:
KNOWLEDGE_BASE.append(("FIFA 2026", "This is hte latest version of the FIFA Worl cup and I am expecting Spain to win it"))

In [63]:
ask_rag( "FIFA")

'Based on the context provided, FIFA 2026 is described as the latest version of the FIFA World Cup, and there is an expectation that Spain will win it [FIFA 2026].'

In [64]:
ask_rag( "FIFA")

'Based on the provided context, FIFA 2026 is described as the latest version of the FIFA World Cup, and the context author expresses an expectation that Spain will win it [FIFA 2026].'

In [65]:
ask_rag( "FIFA")

'Based on the provided context, FIFA 2026 is described as the latest version of the FIFA World Cup, and there is an expectation that Spain will win it [FIFA 2026].'

In [ ]:
from langchain_core

### Connecting LangChain to LM Studio

1.  **Start LM Studio:** Launch LM Studio on your local machine.
2.  **Load a Model:** Download and load a model within LM Studio.
3.  **Start Local Server:** In LM Studio, navigate to the "Local Server" tab and click "Start Server." Note the address, usually `http://localhost:1234/v1`.
4.  **Configure `ChatOpenAI`:** Use the `ChatOpenAI` class in LangChain, setting the `base_url` to your LM Studio server's address and `api_key` to a placeholder (LM Studio doesn't require a real API key for local access).

In [106]:
# Assuming LM Studio is running on your local machine at its default port
lm_studio_base_url = "https://viably-subchorioid-amir.ngrok-free.dev/v1"

# LM Studio often accepts any string for api_key when running locally
lm_studio_api_key = "lm-studio"

Modeal_LM = "qwen2.5-coder-7b-instruct"

lm_studio_llm = ChatOpenAI(
    temperature=0,
    api_key=lm_studio_api_key,
    base_url=lm_studio_base_url,
    model = Modeal_LM
)

print(lm_studio_llm)

metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}} output_version=None client=<openai.resources.chat.completions.completions.Completions object at 0x7b9808e59e80> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7b9808edbf80> root_client=<openai.OpenAI object at 0x7b9808edb260> root_async_client=<openai.AsyncOpenAI object at 0x7b9808ed9d90> model_name='qwen2.5-coder-7b-instruct' temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://viably-subchorioid-amir.ngrok-free.dev/v1' openai_proxy=None stream_chunk_timeout=120.0


In [73]:
# Example usage with the LM Studio LLM
from langchain_core.messages import HumanMessage

response = lm_studio_llm.invoke([HumanMessage(content="Tell me a short story about a brave knight.")])
print(response.content)

Once upon a time in the land of Eldoria, there lived a young and valiant knight named Sir Cedric. He was known throughout the kingdom for his unwavering courage and his noble heart.

One fateful day, a dark sorcerer named Malakai emerged from the shadows, seeking to plunge Eldoria into eternal darkness. His evil magic had already consumed several villages, leaving destruction in its wake. The people of Eldoria turned to Sir Cedric, hoping he could stop this malevolent force.

Sir Cedric set out on his quest with a group of loyal knights and brave warriors. They traveled through treacherous forests, crossed raging rivers, and climbed steep mountains. Along the way, they faced many challenges and battles, but Sir Cedric never faltered in his resolve.

As they drew closer to Malakai's stronghold, the sorcerer revealed himself as a former knight who had been corrupted by power. He offered to surrender if Sir Cedric spared his life. However, Sir Cedric knew that mercy was not an option when

### Steps to Set up `ngrok` for LM Studio (Perform on your LOCAL machine)

1.  **Download and Install `ngrok`:**
    *   Go to the [ngrok website](https://ngrok.com/download) and download the appropriate version for your operating system.
    *   Unzip the downloaded file. You'll get an `ngrok` executable.

2.  **Get your `ngrok` Authtoken:**
    *   If you don't have an `ngrok` account, sign up on the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).
    *   You'll find your authtoken there. It looks something like `ngrok authtoken <YOUR_AUTH_TOKEN>`.

3.  **Authenticate `ngrok` on your local machine:**
    *   Open a terminal or command prompt on your local machine.
    *   Navigate to the directory where you unzipped the `ngrok` executable.
    *   Run the command provided on your ngrok dashboard (e.g., `ngrok authtoken <YOUR_AUTH_TOKEN>`). This will save your authtoken locally.

4.  **Start LM Studio and its Local Server:**
    *   Open LM Studio on your local machine.
    *   Load your desired model.
    *   Go to the "Local Server" tab (the chat icon on the left).
    *   Click "Start Server." Ensure it's running, typically on `http://localhost:1234`.

5.  **Start the `ngrok` Tunnel:**
    *   In your local terminal (where you ran the authtoken command), execute the following command. This will create a tunnel to the port LM Studio is using (default 1234).
        ```bash
        ./ngrok http 1234
        ```
        *(If you're on Windows, it might be `ngrok http 1234` without `./`)*

6.  **Copy the `ngrok` Public URL:**
    *   `ngrok` will output information to your terminal, including a forwarding URL. It will look something like `https://<random_string>.ngrok-free.app`.
    *   Copy this `https://` URL. This is the `base_url` you'll use in your Colab notebook.

Once you have the `ngrok` public URL from your local machine, you can update the `lm_studio_base_url` in your Colab notebook to use this URL. Remember to append `/v1` to the ngrok URL to match the OpenAI API compatibility endpoint.

In [ ]:
# Replace this with your actual ngrok public URL (e.g., 'https://abcd.ngrok-free.app/v1')
# Make sure to append /v1 to the ngrok URL
ngrok_public_url = "https://viably-subchorioid-amir.ngrok-free.dev/v1"

# Re-initialize lm_studio_llm with the ngrok URL
lm_studio_llm = ChatOpenAI(
    temperature=0,
    api_key="lm-studio", # LM Studio often accepts any string for api_key locally
    base_url=ngrok_public_url
)

print(lm_studio_llm)

# You can then try invoking it again:
# from langchain_core.messages import HumanMessage
# response = lm_studio_llm.invoke([HumanMessage(content="Tell me a short story about a brave knight.")])
# print(response.content)

For building a RAG application, you'll still leverage other LangChain components that you've already imported or defined, such as:

*   `ChatPromptTemplate` for structuring your prompts.
*   `StrOutputParser` for processing the model's output.
*   Your `retrieve` function for fetching relevant context from your `KNOWLEDGE_BASE`.
*   Your `ask_rag` function to orchestrate the RAG flow (retrieval + generation).

In [107]:
open_book_prompt_local_LM = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grounded assistant. Use ONLY the information in the CONTEXT to answer. "
     "Cite the source id in square brackets after each fact, e.g. [HR-2024-17 §7.3]. "
     "If the answer is not in the context, reply exactly: "
     "\"I don't have enough information to answer that.\""),
    ("human", "CONTEXT:\n{context}\n\nQUESTION: {question}")
])

In [108]:
open_book_local_LM = open_book_prompt_local_LM | lm_studio_llm | StrOutputParser()

In [112]:
def ask_rag_local_LM(question: str, k: int = 2)-> str:
  context = retrieve(question, k = k )
  print(f"Retrieved context for '{question}': {context}") # Debug print
  answer = open_book_local_LM.invoke({"context":context, "question":  question })
  return answer

In [118]:
ask_rag_local_LM("what does knowledge base say about  FIFA", 2)

Retrieved context for 'FIFA worl': [('FIFA 2026', 'This is hte latest version of the FIFA Worl cup and I am expecting Spain to win it')]


"I don't have enough information to answer that."

In [116]:
ask_rag_local_LM("Please share info about the project Inceptez")

Retrieved context for 'Please share info about the project Inceptez': [('Project Inceptez — sync notes 2024-06-10', 'Project Inceptez engineering sync (Mon 2024-06-10): the team standardized on Claude Opus for the search service. Main reason: best-in-class metadata information under ~50M vectors, which the claims ACL model requires. pgvector was considered but rejected because per-tenant filtering at our recall target was slower in testing.')]


'Project Inceptez is a team working on standardizing Claude Opus for their search service [HR-2024-17 §7.3]. They chose Claude Opus because it provides best-in-class metadata information under approximately 50M vectors, which is essential for the ACL model they are using. The team considered pgvector but rejected it due to slower per-tenant filtering at their recall target during testing [HR-2024-17 §7.3].'

### Verifying the `retrieve` function output

In [113]:
print('--- Testing retrieve("FIFA") ---')
retrieved_context_fifa = retrieve("FIFA")
print(f"Retrieved context for 'FIFA': {retrieved_context_fifa}")

print('\n--- Testing retrieve("Inceptez") ---')
retrieved_context_inceptez = retrieve("Inceptez")
print(f"Retrieved context for 'Inceptez': {retrieved_context_inceptez}")

# Now, let's try invoking the open_book_local_LM directly with this context to further debug
print('\n--- Invoking LM with retrieved context for FIFA ---')
if retrieved_context_fifa:
    response_fifa = open_book_local_LM.invoke({"context": retrieved_context_fifa, "question": "FIFA"})
    print(f"Response for FIFA with retrieved context: {response_fifa}")
else:
    print("No context retrieved for FIFA, so LLM won't be able to answer.")

print('\n--- Invoking LM with retrieved context for Inceptez ---')
if retrieved_context_inceptez:
    response_inceptez = open_book_local_LM.invoke({"context": retrieved_context_inceptez, "question": "Please share info about the project Inceptez"})
    print(f"Response for Inceptez with retrieved context: {response_inceptez}")
else:
    print("No context retrieved for Inceptez, so LLM won't be able to answer.")

--- Testing retrieve("FIFA") ---
Retrieved context for 'FIFA': [('FIFA 2026', 'This is hte latest version of the FIFA Worl cup and I am expecting Spain to win it')]

--- Testing retrieve("Inceptez") ---
Retrieved context for 'Inceptez': [('Project Inceptez — sync notes 2024-06-10', 'Project Inceptez engineering sync (Mon 2024-06-10): the team standardized on Claude Opus for the search service. Main reason: best-in-class metadata information under ~50M vectors, which the claims ACL model requires. pgvector was considered but rejected because per-tenant filtering at our recall target was slower in testing.')]

--- Invoking LM with retrieved context for FIFA ---
Response for FIFA with retrieved context: I don't have enough information to answer that.

--- Invoking LM with retrieved context for Inceptez ---
Response for Inceptez with retrieved context: Project Inceptez is a project where the team standardized on Claude Opus for the search service due to its best-in-class metadata informati

In [119]:
ask("summarize the key findings of 2021 research paper 'RAG Beats'"
"Memory: A study of grounded language model by Noor and Naz")

'There is **no known 2021 research paper titled "RAG Beats Memory: A study of grounded language model" authored by "Noor and Naz."** This title and author combination does not exist in the academic record (arXiv, ACL Anthology, NeurIPS, ICML, ICLR proceedings, or Google Scholar).\n\nYou are likely conflating the **seminal RAG paper** with different authors/details. The foundational paper for Retrieval-Augmented Generation is:\n\n### **"Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"**\n*   **Authors:** Patrick Lewis, Ethan Perez, Aleksandra Piktus, Fabio Petroni, Vladimir Karpukhin, Naman Goyal, Heinrich Küttler, Mike Lewis, Wen-tau Yih, Tim Rocktäschel, Sebastian Riedel, Douwe Kiela (Facebook AI Research / FAIR).\n*   **Venue:** NeurIPS 2020 (arXiv:2005.11401v4, 2021).\n\n---\n\n### **Key Findings of the Actual RAG Paper (Lewis et al., 2020/2021)**\n\nIf this is the paper you intended to reference, here are its definitive key findings:\n\n#### 1. **RAG Outperforms Pu

In [120]:
ask("What is the cut off date for your knowledge")

'My knowledge cutoff is **February 2025**.'